# 🚀 Colab Turbo Manager

### 📱 Mobile-Friendly Drive & Download Tool

#### How to use:
1. Run all cells (Runtime → Run all)
2. Fill inputs in each section
3. Follow on-screen progress bars and messages

✔ No coding required
✔ All sections are collapsible
✔ Works on mobile


In [ ]:
#@title 🔗 Setup (Mount Drive & Install Dependencies)

from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

try:
    import requests, tqdm
except:
    install("requests")
    install("tqdm")

print("✅ Setup complete")


In [ ]:
#@title 📊 Drive Storage Info

import os

def hr(x):
    for u in ['B','KB','MB','GB','TB']:
        if x < 1024:
            return f"{x:.2f} {u}"
        x/=1024

st = os.statvfs("/content/drive/MyDrive")
total = st.f_blocks * st.f_frsize
free = st.f_bavail * st.f_frsize
used = total - free

print("Total:", hr(total))
print("Used :", hr(used))
print("Free :", hr(free))

if free < 2*(1024**3):
    print("⚠️ Low storage")
else:
    print("✅ Storage OK")


In [ ]:
#@title 📁 Drive Management (with Progress)

INPUT_PATH = ""  #@param {type:"string"}
OUTPUT_PATH = ""  #@param {type:"string"}
ACTION = "copy" #@param ["copy","move","rename","delete","compress"]
CONFIRM_DELETE = False  #@param {type:"boolean"}

import shutil, os, zipfile
from tqdm import tqdm

def copy_with_progress(src, dst):
    total = os.path.getsize(src)
    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        with tqdm(total=total, unit='B', unit_scale=True, desc="📁 Copying") as pbar:
            while True:
                buf = fsrc.read(1024 * 1024)
                if not buf:
                    break
                fdst.write(buf)
                pbar.update(len(buf))

try:
    if ACTION == "copy":
        copy_with_progress(INPUT_PATH, OUTPUT_PATH)
        print("✅ Copied")

    elif ACTION == "move":
        copy_with_progress(INPUT_PATH, OUTPUT_PATH)
        os.remove(INPUT_PATH)
        print("✅ Moved")

    elif ACTION == "rename":
        print("✏️ Renaming...")
        os.rename(INPUT_PATH, OUTPUT_PATH)
        print("✅ Renamed")

    elif ACTION == "delete":
        if not CONFIRM_DELETE:
            print("⚠️ Enable CONFIRM_DELETE")
        else:
            print("🗑 Deleting...")
            os.remove(INPUT_PATH)
            print("✅ Deleted")

    elif ACTION == "compress":
        zip_path = OUTPUT_PATH if OUTPUT_PATH.endswith(".zip") else OUTPUT_PATH + ".zip"

        if os.path.isdir(INPUT_PATH):
            file_list = []
            for root, dirs, files in os.walk(INPUT_PATH):
                for f in files:
                    file_list.append(os.path.join(root, f))

            with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
                for file in tqdm(file_list, desc="📦 Compressing"):
                    arcname = os.path.relpath(file, INPUT_PATH)
                    z.write(file, arcname)

            print("✅ Folder compressed:", zip_path)

        elif os.path.isfile(INPUT_PATH):
            with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
                z.write(INPUT_PATH, os.path.basename(INPUT_PATH))
            print("✅ File compressed:", zip_path)

        else:
            print("❌ Invalid path")

except Exception as e:
    print("❌ Error:", e)


In [ ]:
#@title 🔧 Download Settings

URL = ""  #@param {type:"string"}
NAME = "file.bin"  #@param {type:"string"}
THREADS = 6  #@param {type:"integer"}

print("✅ Download settings ready")


In [ ]:
#@title 🚀 Download (with Progress)

import requests, math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

dest = Path("/content/drive/MyDrive/Downloads")
dest.mkdir(exist_ok=True)
file = dest / NAME

def get_size(u):
    try:
        return int(requests.head(u).headers.get("content-length",0))
    except:
        return 0

size = get_size(URL)
pbar = tqdm(total=size if size else None, unit='B', unit_scale=True)

def part(i,s,e):
    h = {"Range": f"bytes={s}-{e}"} if e else {}
    with requests.get(URL, headers=h, stream=True) as r:
        with open(dest/f"p{i}","wb") as f:
            for c in r.iter_content(1024*512):
                if c:
                    f.write(c)
                    pbar.update(len(c))

if size:
    ps = math.ceil(size/THREADS)
    with ThreadPoolExecutor(max_workers=THREADS) as ex:
        ex.map(lambda i: part(i,i*ps,min(size-1,(i+1)*ps-1)), range(THREADS))

    with open(file,"wb") as out:
        for i in range(THREADS):
            p = dest/f"p{i}"
            if p.exists():
                out.write(open(p,"rb").read())
                p.unlink()
else:
    part(0,0,None)

pbar.close()
print("✅ Download complete:", file)
